# Loans and credit facilities

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb`. Native **`term_loan`** and **`revolving_credit`** JSON cover draws, commitment/usage fees, and delayed-draw style availability windows where modeled.


## Concept

- **Term loan:** fixed or floating margin; amortization rules.
- **Revolver:** deterministic draw/repay paths; **commitment** and **usage** fees on drawn vs undrawn.
- **DDTL:** delayed-draw tranche inside `term_loan` encodes availability windows.

**IRR** (money-weighted return) is the rate that zeros PV of all cashflows—conceptually parallel to **discount margin** / **all-in yield** metrics when exposed.


In [1]:
import json
from pathlib import Path

import sys
sys.path.insert(0, "../..")

from _shared import (
    DEMO_AS_OF,
    print_metrics,
    usd_ois_curve,
    usd_sofr_curve,
    usd_sofr_fixings,
)
from finstack_quant.core.market_data import MarketContext
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import (
    price_instrument_with_metrics,
    validate_instrument_json,
)

print("Imports OK (loans / RCF).")


Imports OK (loans / RCF).


## Instrument JSON

**RCF recovery:** For a **senior secured** revolving facility, a **nonzero recovery rate** (here **70%**) is typical in credit-adjusted PV; `0.0` would overstate loss given default in a secured context.


In [2]:
_NOTEBOOK_DATA = json.loads(Path("data/loans_and_credit_facilities.json").read_text())

AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()

term_loan = _NOTEBOOK_DATA['term_loan']

revolver = _NOTEBOOK_DATA['revolver']

term_json = json.dumps(term_loan)
rev_json = json.dumps(revolver)
for label, js in (("term_loan", term_json), ("revolving_credit", rev_json)):
    validate_instrument_json(js)
    print(label, "JSON OK")

print(
    "IRR concept: solve r with sum(CF_t / (1+r)^t)=0 on the facility cashflow schedule (fees + interest + draws/repays)."
)


term_loan JSON OK
revolving_credit JSON OK
IRR concept: solve r with sum(CF_t / (1+r)^t)=0 on the facility cashflow schedule (fees + interest + draws/repays).


## MarketContext


In [3]:
ois = usd_ois_curve(AS_OF)
sofr = usd_sofr_curve(AS_OF)
mc = MarketContext().insert(ois).insert(sofr)
mc.insert_series(usd_sofr_fixings(AS_OF))
market_json = mc.to_json()
print("market ready")


market ready


## Pricing


In [4]:
for label, js in (("term_loan", term_json), ("revolving_credit", rev_json)):
    out = price_instrument_with_metrics(js, market_json, AS_OF_STR, model="discounting")
    print(label, ValuationResult.from_json(out))


term_loan ValuationResult(id="TERM-LOAN-USD-5Y", price=9725966.0728, currency=USD, metrics=0)
revolving_credit ValuationResult(id="RCF-USD-3Y", price=16309707.8077, currency=USD, metrics=1)


## Metrics


In [5]:
for label, js, metrics in (
    ("term_loan", term_json, ["dv01", "effective_rate"]),
    ("revolving_credit", rev_json, ["dv01", "discount_margin", "effective_rate"]),
):
    out = price_instrument_with_metrics(
        js, market_json, AS_OF_STR, model="discounting", metrics=metrics
    )
    vr = ValuationResult.from_json(out)
    print("—", label)
    print_metrics(vr, metrics)


— term_loan
  dv01: -2915.91847172752
— revolving_credit
  dv01: -430.9987824503333


## Takeaways

- **Term loans** and **RCFs** share the same JSON pricing pipeline as simpler rates instruments.
- **Fee stacks** (commitment / usage / facility) interact with draws to shape IRR and PV.
- If a schema field moves between releases, **`validate_instrument_json`** is the first line of defense.
